In [1]:
# PMI-Based Word Similarity - Custom Implementation (No External Libraries)

import pandas as pd
import numpy as np
import re
import math
import random
from collections import defaultdict, Counter
from sklearn.metrics.pairwise import cosine_similarity

# Load your tweet dataset (replace with actual file path or DataFrame)
df = pd.read_csv("TweetSentiment.csv", encoding="ISO-8859-1")
texts = df['text'].dropna().astype(str).tolist()

# ---------------------------
# Step 1: Tokenization
# ---------------------------
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

# Tokenize all texts
tokenized_texts = [tokenize(text) for text in texts]

# ---------------------------
# Step 2: Co-occurrence Counts (window size = 1)
# ---------------------------
word_counts = Counter()
pair_counts = defaultdict(int)
total_windows = 0

for tokens in tokenized_texts:
    for i, word in enumerate(tokens):
        word_counts[word] += 1
        if i > 0:
            pair = (word, tokens[i - 1])
            pair_counts[pair] += 1
            total_windows += 1
        if i < len(tokens) - 1:
            pair = (word, tokens[i + 1])
            pair_counts[pair] += 1
            total_windows += 1

# ---------------------------
# Step 3: Compute PMI Matrix
# ---------------------------
unique_words = list(word_counts.keys())
word_to_index = {word: i for i, word in enumerate(unique_words)}
vocab_size = len(unique_words)

# Initialize PMI matrix
pmi_matrix = np.zeros((vocab_size, vocab_size))

for (w1, w2), count in pair_counts.items():
    if word_counts[w1] > 1 and word_counts[w2] > 1:
        pmi = math.log((count * total_windows) / (word_counts[w1] * word_counts[w2]))
        if pmi > 0:
            i, j = word_to_index[w1], word_to_index[w2]
            pmi_matrix[i][j] = pmi

# ---------------------------
# Step 4: Compute Similar Words for Random Terms
# ---------------------------
# Choose 10 frequent words
frequent_words = [w for w in word_counts if word_counts[w] >= 10]
random_words = random.sample(frequent_words, 10)

# Compute cosine similarity
similarities = cosine_similarity(pmi_matrix)

# Print top 5 similar words for each random word
for word in random_words:
    print(f"\n\033[1mWord:\033[0m {word}")
    idx = word_to_index[word]
    sim_scores = similarities[idx]
    top_indices = sim_scores.argsort()[::-1][1:6]  # exclude itself
    similar_words = [unique_words[i] for i in top_indices]
    print("Most Similar:", ", ".join(similar_words))



Word: like
Most Similar: is, better, and, good, do

Word: happiness
Most Similar: illness, philosophy, transformers, inspite, bh

Word: portfolio
Most Similar: uploaded, galore, phoenixfm, scratched, scroll

Word: getting
Most Similar: get, gettin, losing, following, digging

Word: airport
Most Similar: san, lovin, calcutta, prize, indians

Word: st
Most Similar: aloud, stronger, siya, archie, kayo

Word: g
Most Similar: quoted, er, gooooooood, mitchel, ame

Word: window
Most Similar: murdered, teehee, newt, alyssa, bldg

Word: fell
Most Similar: tommy, puked, listened, falls, clicked

Word: fix
Most Similar: sort, shave, gary, switching, expansion
